In [0]:
# Gold: schema + dimensões enriquecidas
spark.sql("CREATE SCHEMA IF NOT EXISTS voebem.gold")

# dim_empresas: apenas operadores com ICAO (unicos que aparecem no VRA)
spark.sql("""
CREATE OR REPLACE TABLE voebem.gold.dim_empresas AS
SELECT
    icao,
    sigla_iata,
    razao_social,
    servico,
    cidade,
    uf,
    situacao,
    origem_cadastro,
    _ingerido_em,
    _transformado_em
FROM voebem.silver.empresas
WHERE icao IS NOT NULL AND icao != ''
""")

# codigos_operacao: tabela de referencia (DI e tipo de linha)
spark.sql("""
CREATE OR REPLACE TABLE voebem.gold.codigos_operacao AS
SELECT dominio, codigo, descricao, _transformado_em
FROM voebem.silver.codigos_operacao
""")

# dim_aerodromos: enriquecida com coordenadas decimais convertidas de DMS
spark.sql("""
CREATE OR REPLACE TABLE voebem.gold.dim_aerodromos AS
SELECT
    codigo_oaci,
    ciad,
    nome,
    municipio,
    uf_nome,
    municipio_servido,
    uf_servido_nome,
    latitude_dms,
    longitude_dms,
    (
        try_cast(regexp_extract(latitude_dms, '^([0-9]+)', 1) AS DOUBLE) +
        try_cast(regexp_extract(latitude_dms, "([0-9]+)'", 1) AS DOUBLE) / 60.0 +
        try_cast(regexp_extract(latitude_dms, '([0-9]+)"', 1) AS DOUBLE) / 3600.0
    ) *
    CASE WHEN regexp_extract(latitude_dms, '([NS])$', 1) = 'S' THEN -1.0 ELSE 1.0 END
        AS latitude_dec,
    (
        try_cast(regexp_extract(longitude_dms, '^([0-9]+)', 1) AS DOUBLE) +
        try_cast(regexp_extract(longitude_dms, "([0-9]+)'", 1) AS DOUBLE) / 60.0 +
        try_cast(regexp_extract(longitude_dms, '([0-9]+)"', 1) AS DOUBLE) / 3600.0
    ) *
    CASE WHEN regexp_extract(longitude_dms, '([EW])$', 1) = 'W' THEN -1.0 ELSE 1.0 END
        AS longitude_dec,
    altitude_m,
    operacao_diurna,
    operacao_noturna,
    situacao,
    _ingerido_em,
    _transformado_em
FROM voebem.silver.aerodromos
""")

display(spark.sql("""
    SELECT 'dim_empresas' AS tabela, COUNT(*) AS linhas FROM voebem.gold.dim_empresas
    UNION ALL SELECT 'dim_aerodromos', COUNT(*) FROM voebem.gold.dim_aerodromos
    UNION ALL SELECT 'codigos_operacao', COUNT(*) FROM voebem.gold.codigos_operacao
"""))

tabela,linhas
dim_empresas,169
dim_aerodromos,496
codigos_operacao,13


In [0]:
# Gold: fct_voos — deduplicada, enriquecida e com flags de qualidade
# Tratamentos aplicados:
#   1. Deduplicacao por (empresa, voo, data, origem, destino) mantendo _ingerido_em mais recente
#   2. LEFT JOINs com dimensoes (preserva voos sem match — ~0,2% sem empresa, ~10% sem aeroporto estrangeiro)
#   3. Flags de qualidade: tem_prevista, tem_partida_real, tem_chegada_real, atraso_suspeito, recuperou_tempo
#   4. Metricas derivadas: pontualidade_partida, pontualidade_chegada (limiar 15 min)
#   5. Coluna codigo_justificativa descartada (sempre vazia desde abril/2020)

spark.sql("""
CREATE OR REPLACE TABLE voebem.gold.fct_voos AS
WITH dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY icao_empresa_aerea, numero_voo,
                     COALESCE(partida_prevista_data, partida_real_data),
                     icao_aerodromo_origem, icao_aerodromo_destino
            ORDER BY _ingerido_em DESC
        ) AS _rn
    FROM voebem.silver.vra
)
SELECT
    d.icao_empresa_aerea,
    e.razao_social  AS nome_empresa,
    e.sigla_iata    AS sigla_iata_empresa,
    d.numero_voo,
    d.codigo_autorizacao_di,
    codi.descricao  AS desc_di,
    d.codigo_tipo_linha,
    cotl.descricao  AS desc_tipo_linha,
    d.icao_aerodromo_origem,
    ao.nome          AS nome_origem,
    ao.municipio     AS municipio_origem,
    ao.uf_nome       AS uf_origem,
    d.icao_aerodromo_destino,
    ad.nome          AS nome_destino,
    ad.municipio     AS municipio_destino,
    ad.uf_nome       AS uf_destino,
    d.partida_prevista,
    d.partida_prevista_data,
    d.partida_prevista_hora,
    d.partida_real,
    d.partida_real_data,
    d.partida_real_hora,
    d.chegada_prevista,
    d.chegada_prevista_data,
    d.chegada_prevista_hora,
    d.chegada_real,
    d.chegada_real_data,
    d.chegada_real_hora,
    d.situacao_voo,
    d.atraso_partida_min,
    d.atraso_chegada_min,
    d.minutos_recuperados,

    -- Flags de qualidade
    d.partida_prevista IS NOT NULL                       AS tem_prevista,
    d.partida_real IS NOT NULL                           AS tem_partida_real,
    d.chegada_real IS NOT NULL                           AS tem_chegada_real,
    (ABS(d.atraso_partida_min) > 720
     OR ABS(d.atraso_chegada_min) > 720)                 AS atraso_suspeito,
    d.minutos_recuperados > 0                            AS recuperou_tempo,

    -- Metricas derivadas (limiar 15 min = padrao ANAC para pontualidade)
    CASE
        WHEN d.atraso_partida_min IS NULL THEN NULL
        WHEN d.atraso_partida_min <= 15 THEN 'PONTUAL'
        ELSE 'ATRASADO'
    END                                                 AS pontualidade_partida,
    CASE
        WHEN d.atraso_chegada_min IS NULL THEN NULL
        WHEN d.atraso_chegada_min <= 15 THEN 'PONTUAL'
        ELSE 'ATRASADO'
    END                                                 AS pontualidade_chegada,
    
    -- Auditoria
    d._arquivo_origem,
    d._ingerido_em,
    current_timestamp()                                 AS _transformado_em
FROM dedup d
LEFT JOIN voebem.gold.dim_empresas e
    ON d.icao_empresa_aerea = e.icao
LEFT JOIN voebem.gold.dim_aerodromos ao
    ON d.icao_aerodromo_origem = ao.codigo_oaci
LEFT JOIN voebem.gold.dim_aerodromos ad
    ON d.icao_aerodromo_destino = ad.codigo_oaci
LEFT JOIN voebem.gold.codigos_operacao codi
    ON codi.dominio = 'codigo_di' AND codi.codigo = d.codigo_autorizacao_di
LEFT JOIN voebem.gold.codigos_operacao cotl
    ON cotl.dominio = 'codigo_tipo_linha' AND cotl.codigo = d.codigo_tipo_linha
WHERE d._rn = 1
""")

print("fct_voos criada")
display(spark.sql("SELECT COUNT(*) AS total_linhas FROM voebem.gold.fct_voos"))

fct_voos criada


total_linhas
1596861


In [0]:
# Gold: documentacao completa — comentarios de coluna, comentarios de tabela e tags

COMENTARIOS_DIM_EMPRESAS = {
    "icao":              "Codigo ICAO de tres letras da empresa aerea. Chave primaria da dimensao.",
    "sigla_iata":        "Sigla IATA de duas letras, quando publicada pela ANAC.",
    "razao_social":      "Razao social da empresa aerea. Exibida ao consumidor final.",
    "servico":           "Tipo de servico autorizado pela ANAC: transporte regular, nao regular, aeroagricola, taxi aereo.",
    "cidade":            "Municipio da sede ou representante legal.",
    "uf":                "Sigla da unidade federativa da sede.",
    "situacao":          "Situacao do registro na ANAC. Registro inativo permanece porque a empresa pode ter voado no periodo analisado.",
    "origem_cadastro":   "Origem do cadastro: nacional ou estrangeira.",
    "_ingerido_em":      "Auditoria: momento da ingestao no bronze.",
    "_transformado_em":  "Auditoria: momento da construcao da gold.",
}

COMENTARIOS_DIM_AERODROMOS = {
    "codigo_oaci":       "Codigo ICAO (OACI) do aerodromo. Chave primaria da dimensao.",
    "ciad":              "Codigo de identificacao do aerodromo no cadastro da ANAC.",
    "nome":              "Nome do aerodromo como publicado pela ANAC.",
    "municipio":         "Municipio onde o aerodromo esta fisicamente localizado.",
    "uf_nome":           "Nome da UF por extenso (Acre, Sao Paulo), no formato da ANAC.",
    "municipio_servido": "Municipio principal atendido pelo aerodromo, que pode diferir do municipio onde ele fica.",
    "uf_servido_nome":   "Nome por extenso da UF do municipio servido.",
    "latitude_dms":      "Latitude em graus, minutos e segundos (formato ANAC: DD MM SS H).",
    "longitude_dms":     "Longitude em graus, minutos e segundos (formato ANAC: DDD MM SS H).",
    "latitude_dec":      "Latitude convertida para graus decimais. Negativa para Sul. Pronta para uso em mapas e analise espacial.",
    "longitude_dec":     "Longitude convertida para graus decimais. Negativa para Oeste. Pronta para uso em mapas e analise espacial.",
    "altitude_m":         "Altitude do aerodromo em metros.",
    "operacao_diurna":   "Indica se o aerodromo esta autorizado a operar em horario diurno.",
    "operacao_noturna":  "Indica se o aerodromo esta autorizado a operar em horario noturno.",
    "situacao":          "Situacao do aerodromo no cadastro da ANAC: Cadastrado ou Interditado.",
    "_ingerido_em":      "Auditoria: momento da ingestao no bronze.",
    "_transformado_em":  "Auditoria: momento da construcao da gold.",
}

COMENTARIOS_CODIGOS = {
    "dominio":           "A qual coluna do VRA este codigo pertence: codigo_di ou codigo_tipo_linha.",
    "codigo":            "O codigo como aparece no VRA.",
    "descricao":         "Descricao oficial do codigo, curada da pagina de descricao de variaveis da ANAC.",
    "_transformado_em":  "Auditoria: momento da construcao da gold.",
}

COMENTARIOS_FCT_VOOS = {
    "icao_empresa_aerea":       "Codigo ICAO de tres letras da empresa aerea. Chave para dim_empresas.",
    "nome_empresa":             "Razao social da empresa aerea, trazida via LEFT JOIN com dim_empresas. NULL se a empresa nao constar no cadastro (~0,2% das etapas).",
    "sigla_iata_empresa":       "Sigla IATA da empresa, trazida via LEFT JOIN com dim_empresas.",
    "numero_voo":               "Numero do voo divulgado pela companhia. Identificador comercial, nao numerico: pode ter zero a esquerda.",
    "codigo_autorizacao_di":    "Codigo de autorizacao (DI) da etapa. Descricao em desc_di.",
    "desc_di":                  "Descricao do codigo DI (Etapa Regular, Extra, Retorno, Charter, etc.), trazida via LEFT JOIN com codigos_operacao.",
    "codigo_tipo_linha":        "Codigo do tipo de linha (N, C, I, G). Descricao em desc_tipo_linha.",
    "desc_tipo_linha":         "Descricao do tipo de linha (Domestica Mista, Internacional Cargueira, etc.), trazida via LEFT JOIN com codigos_operacao.",
    "icao_aerodromo_origem":   "Codigo ICAO do aerodromo de origem. Chave para dim_aerodromos.",
    "nome_origem":              "Nome do aerodromo de origem, trazido via LEFT JOIN com dim_aerodromos. NULL para aeroportos estrangeiros (~10%).",
    "municipio_origem":         "Municipio do aerodromo de origem, trazido via LEFT JOIN com dim_aerodromos.",
    "uf_origem":                "Nome da UF do aerodromo de origem, trazido via LEFT JOIN com dim_aerodromos.",
    "icao_aerodromo_destino":  "Codigo ICAO do aerodromo de destino. Chave para dim_aerodromos.",
    "nome_destino":             "Nome do aerodromo de destino, trazido via LEFT JOIN com dim_aerodromos. NULL para aeroportos estrangeiros.",
    "municipio_destino":        "Municipio do aerodromo de destino, trazido via LEFT JOIN com dim_aerodromos.",
    "uf_destino":               "Nome da UF do aerodromo de destino, trazido via LEFT JOIN com dim_aerodromos.",
    "partida_prevista":         "Horario de partida programado pela companhia, na hora local do aeroporto de origem. NULL em ~3% dos voos REALIZADO.",
    "partida_prevista_data":    "Data da partida programada. Usada para liquid clustering.",
    "partida_prevista_hora":    "Hora e minuto da partida programada (HH:mm), separada para analise por faixa horaria.",
    "partida_real":             "Horario em que a aeronave efetivamente saiu. NULL em voo cancelado.",
    "partida_real_data":        "Data da partida efetiva.",
    "partida_real_hora":        "Hora e minuto da partida efetiva (HH:mm).",
    "chegada_prevista":         "Horario de chegada programado, na hora local do aeroporto de destino.",
    "chegada_prevista_data":    "Data da chegada programada.",
    "chegada_prevista_hora":    "Hora e minuto da chegada programada (HH:mm).",
    "chegada_real":             "Horario em que a aeronave efetivamente pousou. NULL em voo cancelado.",
    "chegada_real_data":        "Data da chegada efetiva.",
    "chegada_real_hora":        "Hora e minuto da chegada efetiva (HH:mm).",
    "situacao_voo":             "Situacao informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando nao.",
    "atraso_partida_min":       "Minutos entre a partida programada e a partida efetiva. Positivo e atraso, negativo e antecipacao. NULL quando partida_prevista e ausente.",
    "atraso_chegada_min":       "Minutos entre a chegada programada e a chegada efetiva. Positivo e atraso, negativo e antecipacao.",
    "minutos_recuperados":      "Minutos recuperados em voo: atraso de partida menos atraso de chegada. Positivo significa que chegou menos atrasada do que saiu.",
    "tem_prevista":             "Flag de qualidade: TRUE se partida_prevista nao e NULL. FALSE indica que nao ha horario programado para calcular atraso.",
    "tem_partida_real":         "Flag de qualidade: TRUE se partida_real nao e NULL. FALSE indica voo cancelado.",
    "tem_chegada_real":         "Flag de qualidade: TRUE se chegada_real nao e NULL. FALSE indica voo cancelado.",
    "atraso_suspeito":          "Flag de qualidade: TRUE se ABS(atraso_partida_min) > 720 min (12h) OU ABS(atraso_chegada_min) > 720. Indica provavel erro de digitacao ou reescalonamento cross-day.",
    "recuperou_tempo":          "Flag de qualidade: TRUE se minutos_recuperados > 0. A aeronave compensou atraso durante o voo.",
    "pontualidade_partida":     "Metrica derivada: PONTUAL se atraso_partida_min <= 15 min, ATRASADO se > 15. NULL quando atraso e indisponivel. Limiar 15 min e o padrao ANAC.",
    "pontualidade_chegada":     "Metrica derivada: PONTUAL se atraso_chegada_min <= 15 min, ATRASADO se > 15. NULL quando atraso e indisponivel.",
    "_arquivo_origem":          "Auditoria: nome do arquivo CSV mensal da ANAC de onde a linha veio.",
    "_ingerido_em":             "Auditoria: momento em que a linha entrou no bronze.",
    "_transformado_em":         "Auditoria: momento em que a gold foi construida a partir da silver.",
}

TABELAS_GOLD = {
    "voebem.gold.fct_voos": (
        "Gold - fato de etapas de voo deduplicada e enriquecida. Uma linha por etapa de voo real. "
        "Tratamentos aplicados: deduplicacao por (empresa, voo, data, origem, destino) mantendo "
        "o _ingerido_em mais recente; enriquecimento com nomes de empresa, aerodromo e codigos de operacao via LEFT JOIN; "
        "flags de qualidade (tem_prevista, atraso_suspeito, recuperou_tempo); metricas derivadas (pontualidade_partida, pontualidade_chegada). "
        "Coluna codigo_justificativa descartada (sempre vazia desde abril/2020). Liquid clustering em partida_prevista_data.",
        {"camada": "gold", "dominio": "aviacao", "fonte": "ANAC-VRA", "grao": "etapa_de_voo"},
    ),
    "voebem.gold.dim_empresas": (
        "Gold - dimensao de empresas aereas. Contem apenas operadores com codigo ICAO (169 de 879), "
        "que sao os que de fato aparecem no VRA. Inclui registros inativos para nao perder empresas que operaram "
        "no periodo historico.",
        {"camada": "gold", "dominio": "aviacao", "fonte": "ANAC-Operador-Aereo", "grao": "empresa"},
    ),
    "voebem.gold.dim_aerodromos": (
        "Gold - dimensao de aerodromos brasileiros. Enriquecida com latitude_dec e longitude_dec convertidas "
        "do formato DMS original da ANAC para graus decimais, prontas para analise espacial. "
        "Cobre apenas aerodromos brasileiros — aeroportos estrangeiros do VRA nao constam (propriedade da fonte).",
        {"camada": "gold", "dominio": "aviacao", "fonte": "ANAC-Aerodromos", "grao": "aerodromo"},
    ),
    "voebem.gold.codigos_operacao": (
        "Gold - tabela de referencia de codigos de operacao (DI e tipo de linha) com as descricoes oficiais da ANAC.",
        {"camada": "gold", "dominio": "aviacao", "fonte": "ANAC-seed", "grao": "codigo"},
    ),
}

for tabela, mapa in [
    ("voebem.gold.dim_empresas",     COMENTARIOS_DIM_EMPRESAS),
    ("voebem.gold.dim_aerodromos",   COMENTARIOS_DIM_AERODROMOS),
    ("voebem.gold.codigos_operacao", COMENTARIOS_CODIGOS),
    ("voebem.gold.fct_voos",         COMENTARIOS_FCT_VOOS),
]:
    for coluna, comentario in mapa.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    print(f"{len(mapa)} colunas comentadas em {tabela}")

for tabela, (comentario, tags) in TABELAS_GOLD.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    pares = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({pares})")
    print(f"{tabela}: comentario + {len(tags)} tags")

10 colunas comentadas em voebem.gold.dim_empresas
17 colunas comentadas em voebem.gold.dim_aerodromos
4 colunas comentadas em voebem.gold.codigos_operacao
42 colunas comentadas em voebem.gold.fct_voos
voebem.gold.fct_voos: comentario + 4 tags
voebem.gold.dim_empresas: comentario + 4 tags
voebem.gold.dim_aerodromos: comentario + 4 tags
voebem.gold.codigos_operacao: comentario + 4 tags


In [0]:
# Gold: otimizacao — liquid clustering na fact table para queries por data
spark.sql("ALTER TABLE voebem.gold.fct_voos CLUSTER BY (partida_prevista_data)")
print("Liquid clustering ativado em fct_voos (partida_prevista_data)")

# OPTIMIZE para reescrever arquivos e aplicar o clustering imediatamente
spark.sql("OPTIMIZE voebem.gold.fct_voos")
print("OPTIMIZE concluido")

Liquid clustering ativado em fct_voos (partida_prevista_data)
OPTIMIZE concluido


In [0]:
# Gold: verificacao final — contagem, cobertura de enriquecimento e distribuicao das flags

print("=== 1. Silver vs Gold (dedup) ===")
display(spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM voebem.silver.vra)       AS silver,
        (SELECT COUNT(*) FROM voebem.gold.fct_voos)     AS gold,
        (SELECT COUNT(*) FROM voebem.silver.vra)
          - (SELECT COUNT(*) FROM voebem.gold.fct_voos)  AS deduplicadas
"""))

print("=== 2. Cobertura de enriquecimento ===")
display(spark.sql("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN nome_empresa IS NOT NULL THEN 1 ELSE 0 END) AS com_nome_empresa,
        ROUND(100.0 * SUM(CASE WHEN nome_empresa IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_empresa,
        SUM(CASE WHEN nome_origem IS NOT NULL THEN 1 ELSE 0 END) AS com_nome_origem,
        ROUND(100.0 * SUM(CASE WHEN nome_origem IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_origem,
        SUM(CASE WHEN nome_destino IS NOT NULL THEN 1 ELSE 0 END) AS com_nome_destino,
        ROUND(100.0 * SUM(CASE WHEN nome_destino IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_destino
    FROM voebem.gold.fct_voos
"""))

print("=== 3. Distribuicao das flags de qualidade ===")
display(spark.sql("""
    SELECT
        situacao_voo,
        SUM(CASE WHEN tem_prevista THEN 1 ELSE 0 END) AS com_prevista,
        SUM(CASE WHEN NOT tem_prevista THEN 1 ELSE 0 END) AS sem_prevista,
        SUM(CASE WHEN atraso_suspeito THEN 1 ELSE 0 END) AS atraso_suspeito,
        SUM(CASE WHEN recuperou_tempo THEN 1 ELSE 0 END) AS recuperou_tempo
    FROM voebem.gold.fct_voos
    GROUP BY situacao_voo
    ORDER BY situacao_voo
"""))

print("=== 4. Pontualidade (limiar 15 min) ===")
display(spark.sql("""
    SELECT
        pontualidade_partida,
        pontualidade_chegada,
        COUNT(*) AS total
    FROM voebem.gold.fct_voos
    GROUP BY pontualidade_partida, pontualidade_chegada
    ORDER BY total DESC
"""))

print("=== 5. Conversao de coordenadas (amostra dim_aerodromos) ===")
display(spark.sql("""
    SELECT codigo_oaci, nome, latitude_dms, latitude_dec, longitude_dms, longitude_dec
    FROM voebem.gold.dim_aerodromos
    WHERE latitude_dec IS NOT NULL
    LIMIT 5
"""))

=== 1. Silver vs Gold (dedup) ===


silver,gold,deduplicadas
1597255,1596861,394


=== 2. Cobertura de enriquecimento ===


total,com_nome_empresa,pct_empresa,com_nome_origem,pct_origem,com_nome_destino,pct_destino
1596861,1593835,99.81,1434116,89.81,1433471,89.77


=== 3. Distribuicao das flags de qualidade ===


situacao_voo,com_prevista,sem_prevista,atraso_suspeito,recuperou_tempo
CANCELADO,45795,0,0,0
REALIZADO,1505996,45070,1886,1022093


=== 4. Pontualidade (limiar 15 min) ===


pontualidade_partida,pontualidade_chegada,total
PONTUAL,PONTUAL,1199194
ATRASADO,ATRASADO,195640
null,null,90865
ATRASADO,PONTUAL,59844
PONTUAL,ATRASADO,51318


=== 5. Conversao de coordenadas (amostra dim_aerodromos) ===


codigo_oaci,nome,latitude_dms,latitude_dec,longitude_dms,longitude_dec
SBRB,Plácido de Castro,"09°52'06""S",-9.868333333333334,"067°53'53""W",-67.89805555555556
SWPI,Parintins,"02°40'25""S",-2.673611111111111,"056°46'39""W",-56.777499999999996
SBMQ,Alberto Alcolumbre,"00°03'02""N",0.050555555555555555,"051°04'20""W",-51.07222222222222
SBVC,Glauber de Andrade Rocha,"14°54'28""S",-14.907777777777778,"040°54'53""W",-40.914722222222224
SBLE,HORÁCIO DE MATTOS,"12°28'56""S",-12.482222222222223,"041°16'37""W",-41.276944444444446


# Arquitetura da Camada Gold

## Modelo dimensional

| Tabela | Tipo | Grao | Linhas | Origem |
|--------|------|------|--------|--------|
| `fct_voos` | Fato | etapa_de_voo | ~1,6M | silver.vra (deduplicada + enriquecida) |
| `dim_empresas` | Dimensao | empresa | 169 | silver.empresas (apenas com ICAO) |
| `dim_aerodromos` | Dimensao | aerodromo | 496 | silver.aerodromos (com coordenadas decimais) |
| `codigos_operacao` | Referencia | codigo | 13 | silver.codigos_operacao |

## Tratamentos aplicados na gold

1. **Deduplicacao**: `ROW_NUMBER() OVER (PARTITION BY empresa, voo, data, origem, destino ORDER BY _ingerido_em DESC)` — removeu 394 linhas duplicadas (reenvio de arquivo ANAC)
2. **Enriquecimento**: LEFT JOINs com dim_empresas, dim_aerodromos e codigos_operacao — preserva voos sem match (0,19% sem empresa, 10% sem aeroporto estrangeiro)
3. **Flags de qualidade**: `tem_prevista`, `tem_partida_real`, `tem_chegada_real`, `atraso_suspeito` (>12h), `recuperou_tempo`
4. **Metricas de negocio**: `pontualidade_partida` e `pontualidade_chegada` (PONTUAL se atraso <= 15 min, ATRASADO caso contrario) — padrao ANAC
5. **Coordenadas decimais**: conversao DMS -> decimal em dim_aerodromos para analise espacial
6. **Coluna descartada**: `codigo_justificativa` removida (sempre vazia desde abril/2020)

## Boas praticas de engenharia de dados implementadas

* **Naming convention**: `fct_` para fatos, `dim_` para dimensoes
* **Liquid clustering**: `CLUSTER BY (partida_prevista_data)` para queries por data
* **Documentacao 100%**: comentarios de coluna e tabela + tags (camada, dominio, fonte, grao) em todas as tabelas
* **Auditoria completa**: `_ingerido_em` (bronze), `_transformado_em` (gold), `_arquivo_origem` (arquivo ANAC)
* **Tratamento de erro**: `try_cast` na conversao de DMS tolera formatos inesperados
* **Esquema governado**: todas as tabelas em `voebem.gold` com Unity Catalog